In [7]:
import torch

In [8]:
# Définition du MDP
states = [(0,0), (0,1), (1,0), (1,1)]
actions = ['up', 'down', 'left', 'right']
gamma = 0.9
theta = 1e-4

# Mapping état -> index
state_idx = {s:i for i,s in enumerate(states)}
n_states = len(states)
n_actions = len(actions)

# Fonction de transition et récompense
def transition_reward(s, a):
    x, y = s
    if s == (1,1):
        return s, 0
    if a == 'up':
        next_state = (max(x-1,0), y)
    elif a == 'down':
        next_state = (min(x+1,1), y)
    elif a == 'left':
        next_state = (x, max(y-1,0))
    elif a == 'right':
        next_state = (x, min(y+1,1))
    reward = 1 if next_state == (1,1) else 0
    return next_state, reward

# Initialisation des valeurs des états (tenseur PyTorch)
V = torch.zeros(n_states)

# Value Iteration avec PyTorch
while True:
    delta = 0
    for s in states:
        i = state_idx[s]
        v = V[i].item()
        action_values = []
        for a in actions:
            next_s, r = transition_reward(s,a)
            j = state_idx[next_s]
            action_values.append(r + gamma * V[j])
        V[i] = torch.max(torch.tensor(action_values))
        delta = max(delta, abs(v - V[i].item()))
    if delta < theta:
        break

# Extraction de la politique optimale
policy = {}
for s in states:
    if s == (1,1):
        policy[s] = None
    else:
        action_values = {}
        for a in actions:
            next_s, r = transition_reward(s,a)
            j = state_idx[next_s]
            action_values[a] = r + gamma * V[j].item()
        policy[s] = max(action_values, key=action_values.get)

# Affichage
print("Valeurs des états :")
for s in states:
    print(f"{s}: {V[state_idx[s]].item():.2f}")

print("\nPolitique optimale :")
for s in states:
    print(f"{s}: {policy[s]}")


Valeurs des états :
(0, 0): 0.90
(0, 1): 1.00
(1, 0): 1.00
(1, 1): 0.00

Politique optimale :
(0, 0): down
(0, 1): down
(1, 0): right
(1, 1): None


Points importants

Ici V est un tenseur PyTorch, ce qui permet d’utiliser facilement GPU si nécessaire.

torch.max calcule la valeur maximale parmi les actions.

La politique optimale est extraite exactement comme avant, mais à partir du tenseur PyTorch.

Ce code est facilement extensible à des MDP plus grands ou à des approximations par réseaux neuronaux (Deep RL).